# Cantidad neta semanal ERP y corte estricto
Se conserva `cantidad` en la escala registrada por el ERP, sin convertir cajas a unidades, sin sumar `cantidad + unidades` y sin normalizar el target. MAE/RMSE están en esa escala. Los negativos muestran patrones compatibles con anulaciones, devoluciones o ajustes operativos; no se afirma que todos sean devoluciones.

El constructor compartido suma primero todas las transacciones positivas y negativas por sucursal-producto y semana `W-SUN`. Conserva `cantidad_neta_original` y define `cantidad = max(cantidad_neta_original, 0)` para modelar. Un saldo neto negativo no se interpreta como demanda negativa. Los NaN siguen provocando error; sólo se rellenan huecos interiores sin transacciones, bajo el supuesto de continuidad del sistema.

El corte sigue siendo **2026-01-01**. Se excluye de la evaluación la semana 2025-12-29 a 2026-01-04 en todas las series, incluso si sólo registra transacciones de enero. El test comienza con semanas completas posteriores (etiqueta 2026-01-11). Las semanas de los extremos del dataset pueden ser parciales.

ARIMA usa directamente el constructor compartido a partir de las transacciones. Se mantienen ADF para d, p/q=0..3, AIC y mínimos de 20 semanas en el calendario completo, 10 en train y 2 en test evaluable. No se excluyen series por intermitencia.

Se conserva el pronóstico multi-step desde el fin del train. La semana cruzada se pronostica como paso intermedio pero no se puntúa; el 11 de enero corresponde al paso 2 cuando el origen es 28 de diciembre. Las métricas se calculan sobre la concatenación de observaciones de las series modeladas, no promediando métricas por serie. Se exportan claves, origen y horizonte de cada predicción.

In [1]:
import sys
from pathlib import Path
import json
import hashlib
import platform
import importlib.metadata
import pandas as pd
sys.path.insert(0, str(Path('..').resolve()))
from src.Modelos.ArimaPredictor import ArimaPredictor
from src.preprocessing.demanda_semanal import excluir_semana_corte
ruta_datos = Path('../datasets/dataset_maestro_dashboard.csv')
df = pd.read_csv(ruta_datos)
arima_model = ArimaPredictor(df)
metricas_arima = arima_model.entrenar_y_evaluar()
display(pd.DataFrame([metricas_arima]))
display(pd.DataFrame([arima_model.conteo_series]))
salida = Path('../resultados/semanal')
salida.mkdir(parents=True, exist_ok=True)
pd.DataFrame([metricas_arima]).to_csv(salida / 'metricas_arima.csv', index=False)
(salida / 'conteo_arima.json').write_text(json.dumps(arima_model.conteo_series, indent=2), encoding='utf-8')
arima_model.predicciones.to_csv(salida / 'predicciones_arima_concatenadas.csv', index=False)
arima_model.semanal.to_csv(salida / 'dataset_semanal.csv', index=False)
arima_model.semanal_evaluable.to_csv(salida / 'dataset_semanal_evaluable.csv', index=False)
arima_model.semanas_excluidas.to_csv(salida / 'semanas_excluidas_corte.csv', index=False)
assert arima_model.conteo_series['encontradas'] == sum(arima_model.conteo_series[k] for k in ['descartadas','fallidas','modeladas'])
assert (arima_model.predicciones.semana - pd.Timedelta(days=6) >= pd.Timestamp('2026-01-01')).all()
manifesto = {
    'version_target': 'neto_erp_no_negativo_corte_estricto_v1',
    'sha256_dataset': hashlib.sha256(ruta_datos.read_bytes()).hexdigest(),
    'sha256_constructor': hashlib.sha256(Path('../src/preprocessing/demanda_semanal.py').read_bytes()).hexdigest(),
    'protocolo': 'multi-step desde origen fijo', 'frecuencia': 'W-SUN',
    'fecha_corte': '2026-01-01', 'observaciones_evaluadas': len(arima_model.predicciones),
    'observaciones_excluidas_corte': len(arima_model.semanas_excluidas),
    'python': platform.python_version(),
    'versiones': {p: importlib.metadata.version(p) for p in ['pandas','numpy','statsmodels','scikit-learn']}
}
(salida / 'experimento_arima.json').write_text(json.dumps(manifesto, indent=2), encoding='utf-8')
print('Observaciones ARIMA evaluadas:', len(arima_model.predicciones))

Iniciando entrenamiento ARIMA (Iteracion por Sucursal y Producto)...
Semanas excluidas por cruce del corte: 144
Resumen de series ARIMA: {'potenciales': 266, 'encontradas': 240, 'descartadas': 111, 'fallidas': 0, 'modeladas': 129}
Entrenamiento completado.


,Modelo,MAE,RMSE,R2
0,ARIMA (Optimizado ADF/AIC),0.23115,0.572762,0.210971


,potenciales,encontradas,descartadas,fallidas,modeladas
0,266,240,111,0,129


Observaciones ARIMA evaluadas: 3444
